# Stage 11 — Option buying ranking

Full-universe backtest (notebook 17, WITH the 4H hard filter applied) showed 70.6% of qualified instruments with a positive avg R, mean 0.198, median 0.170. This ranks instruments the same way the other four strategies are ranked (`strategy_scoring.py`), so the shortlist is the instruments where this specific signal+filter+lifecycle actually worked, not the whole universe.

**`min_trades=20` here, not the 30 used for debit spread** - the 4H filter roughly halves signal frequency, so per-instrument closed_trades over the same window now tops out around 27 with a mean of ~19; a 30-trade bar would leave nothing ranked. See `score_for_option_buying()`'s docstring for the full reasoning.

median_r_achieved is weighted highest (not avg_r_achieved) - same median-over-mean preference as the rest of this project's stats.

In [7]:
import sys
sys.path.append("..")

import pandas as pd
from src.strategy_scoring import score_for_option_buying, save_ranking

comparison = pd.read_excel("../exports/OPTION_BUYING_COMPARISON.xlsx", sheet_name="Comparison")
comparison.shape

(216, 8)

In [8]:
ranked, watchlist = score_for_option_buying(comparison)
ranked.head(20)

,symbol,option_buying_score,outlier_reliance_flag,closed_trades,win_rate_pct,avg_r_achieved,median_r_achieved,median_candles_held,pct_trades_trailing_activated
0,ADANIENSOL,92.8,,34,64.7,0.50,0.46,5.0,61.8
1,RVNL,91.9,,42,73.8,0.79,0.68,7.0,71.4
2,GMRAIRPORT,91.6,,34,73.5,0.52,0.39,6.0,73.5
3,PFC,91.4,,27,70.4,0.82,0.65,7.0,70.4
4,KPITTECH,90.8,,37,75.7,0.58,0.62,7.0,70.3
5,COFORGE,89.9,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.35R),43,65.1,0.67,0.32,6.0,65.1
6,GODREJPROP,89.8,,32,71.9,0.48,0.32,6.0,68.8
7,SUZLON,89.7,,39,61.5,0.74,0.50,6.0,59.0
8,MAHABANK,89.5,,35,68.6,0.36,0.42,6.0,62.9
9,BDL,89.5,,38,68.4,0.56,0.30,6.0,60.5


In [9]:
# How many top-20 candidates carry the outlier-reliance warning?
print(f"Instruments passing min_trades filter: {len(ranked)} of {len(comparison)}")
print(f"Flagged in top 20: {(ranked.head(20)['outlier_reliance_flag'] != '').sum()}")
ranked[ranked["outlier_reliance_flag"] != ""].head(10)

Instruments passing min_trades filter: 212 of 216
Flagged in top 20: 2


,symbol,option_buying_score,outlier_reliance_flag,closed_trades,win_rate_pct,avg_r_achieved,median_r_achieved,median_candles_held,pct_trades_trailing_activated
5,COFORGE,89.9,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.35R),43,65.1,0.67,0.32,6.0,65.1
18,RECLTD,85.7,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.40R),31,67.7,0.63,0.23,6.0,64.5
21,PETRONET,83.8,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.39R),32,68.8,0.68,0.29,8.5,62.5
25,POWERINDIA,81.1,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.36R),45,64.4,0.61,0.25,7.0,57.8
26,SHRIRAMFIN,80.5,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.61R),38,68.4,0.81,0.20,7.0,63.2
28,OIL,79.0,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.43R),32,62.5,0.66,0.23,7.0,62.5
32,DIVISLAB,77.5,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.38R),30,63.3,0.58,0.20,7.0,60.0
35,PAYTM,77.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.41R),35,62.9,0.61,0.20,7.0,57.1
40,BANKINDIA,75.8,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.33R),32,62.5,0.55,0.22,7.5,62.5
44,INFY,74.1,⚠ RELIES ON OUTLIER WINS (avg-median gap 0.45R),42,61.9,0.65,0.20,8.0,57.1


In [10]:
save_ranking(ranked, "RANKING_option_buying.xlsx")

Saved ranking to /Users/hgmac/Documents/3_trading/Quant/exports/RANKING_option_buying.xlsx


'/Users/hgmac/Documents/3_trading/Quant/exports/RANKING_option_buying.xlsx'

In [11]:
# Instruments below min_trades - not blended into the percentile ranking,
# but not dropped either. Sorted by avg_r_achieved. Re-check at the next
# periodic refresh - trade count grows on its own as more calendar time passes.
print(f"Instruments on watchlist (below min_trades): {len(watchlist)}")
watchlist

Instruments on watchlist (below min_trades): 4


,symbol,closed_trades,win_rate_pct,avg_r_achieved,median_r_achieved,median_candles_held,pct_trades_trailing_activated
0,SWIGGY,18,83.3,1.21,0.94,6.0,77.8
1,WAAREEENER,18,61.1,0.24,0.21,6.0,55.6
2,ATHERENERG,16,62.5,0.14,0.07,5.0,62.5
3,HYUNDAI,16,56.2,-0.11,0.04,8.0,56.2


In [12]:
save_ranking(watchlist, "WATCHLIST_option_buying.xlsx")

Saved ranking to /Users/hgmac/Documents/3_trading/Quant/exports/WATCHLIST_option_buying.xlsx


'/Users/hgmac/Documents/3_trading/Quant/exports/WATCHLIST_option_buying.xlsx'